# Homework 2: Ridge and Lasso Regression on Our Data

In this homework assignment, I will be performing ridge and lasso regression on my datasets. One thing to note is that all three of my datasets have binary targets, so analyzing the accuracy of them will be a lot harder. I will be constructing a decile based system that will look at the performance compared to the actual values.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

## Dataset 1: Instacart Analysis

The first dataset I am using is one that looks at historical instacart orders. This dataset's main goal is predict whether or not someone will order a product. Below I will load in the dataset and also perform some feature engineering. The features I am creating are used to help predict how likely that product is ordered in that specific order.

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [18]:
X = data1_transformed.drop(columns = ['ordered'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

lasso = Lasso()

lasso.fit(X_train, y_train)

y_preds = lasso.predict(X_test)

values = {'Actual' : y_test, 'Predicted' : y_preds}

values = pd.DataFrame(values)

values.head()

,Actual,Predicted
281280,1.0,0.061787
38562,0.0,0.062193
220303,0.0,0.061132
211506,0.0,0.062367
321641,0.0,0.057761


In [19]:
values['Decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

values.head()

,Actual,Predicted,Decile
281280,1.0,0.061787,6
38562,0.0,0.062193,7
220303,0.0,0.061132,5
211506,0.0,0.062367,8
321641,0.0,0.057761,1


In [30]:
decile_table = values.groupby('Decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

decile_table['lift'] = decile_table['avg_ordered']/overall_ordered

decile_table['avg_predicted'] = decile_table['avg_predicted'].map(lambda x: f'{x:.3f}')
decile_table['avg_ordered'] = decile_table['avg_ordered'].map(lambda x : f'{x:.3f}')
decile_table['lift'] = decile_table['lift'].map(lambda x: f'{x:.3f}')

decile_table

,avg_predicted,avg_ordered,lift
Decile,,,
1,0.059,0.059,0.933
2,0.060,0.067,1.048
3,0.060,0.062,0.977
4,0.061,0.060,0.944
5,0.061,0.067,1.057
6,0.062,0.065,1.028
7,0.062,0.065,1.020
8,0.063,0.061,0.960
9,0.063,0.067,1.057


In [32]:
ridge = Ridge()

ridge.fit(X_train, y_train)

y_preds = ridge.predict(X_test)

values = {'Actual' : y_test, 'Predicted' : y_preds}

values = pd.DataFrame(values)

values['Decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

decile_table = values.groupby('Decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

decile_table['lift'] = decile_table['avg_ordered']/overall_ordered

decile_table['avg_predicted'] = decile_table['avg_predicted'].map(lambda x: f'{x:.3f}')
decile_table['avg_ordered'] = decile_table['avg_ordered'].map(lambda x : f'{x:.3f}')
decile_table['lift'] = decile_table['lift'].map(lambda x: f'{x:.3f}')

decile_table

,avg_predicted,avg_ordered,lift
Decile,,,
1,-0.010,0.007,0.107
2,-0.001,0.010,0.162
3,0.007,0.016,0.259
4,0.017,0.019,0.295
5,0.030,0.031,0.488
6,0.046,0.043,0.679
7,0.067,0.055,0.870
8,0.091,0.077,1.204
9,0.128,0.120,1.894


Comparing the two results, the ridge regression performed much better than the lasso regression. It can be seen that the lasso regression lift is pretty uniform for all ten deciles, unlike the ridge regression, which has a steady increase. You can see there is a massive jump in lift from the ninth to the tenth decile which could be due to the large class imbalance.

## Dataset 2: Amazon Reviews Dataset

The second dataset I am using looks into amazon reviews based on products that the platform sells. Due to the large amount of products on Amazon, I am specifically focusing on the grocery items sold on Amazon. I also will be selecting the top 30 categories of products due to the large amount of products as well.

In [4]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked'])
y = data2_transformed['liked']

X.head()

,rating,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,5,False,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,1,True,1,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,4,True,1,0,0,1,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,5,True,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,5,True,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


## Dataset 3: Ecommerce Dataset

The third dataset I am using looks into ecommerce order information. This dataset shows different orders from an ecommerce company. I am attempting to create fake instances of no orders so it can be used in a predictive way of whether or not someone ordered something, but right now I feel like it is very obvious which instances are fake and which are not.

In [5]:
data3 = pd.read_csv('/workspaces/DX799---HW-Assignments/data.csv', encoding = 'windows-1252')
data3['Purchased'] = 1

customer_id = data3['Customer ID'].values
segments = data3['Segment'].values
cities = data3['City'].values
state = data3['State'].values
region = data3['Region'].values
product_id = data3['Product ID'].values
categories = data3['Category'].values
subcategories = data3['Sub-Category'].values

def generate_row(df):
    row = {}

    row['Customer ID'] = np.random.choice(customer_id)
    cust_id = row['Customer ID']
    purchased = set(df[df['Customer ID'] == cust_id]['Product ID'])
    candidates = list(set(product_id) - purchased)

    row['Segment'] = np.random.choice(segments)
    row['State'] = np.random.choice(state)
    row['City'] = np.random.choice(cities)
    row['Region'] = np.random.choice(region)
    row['Product ID'] = np.random.choice(candidates)
    row['Category'] = np.random.choice(categories)
    row['Sub-Category'] = np.random.choice(subcategories)

    row['Sales'] = 0
    row['Discount'] = 0
    row['Quantity'] = 0
    row['Profit'] = 0
    row['Purchased'] = 0

    return row

fake_rows = pd.DataFrame([generate_row(data3) for n in range(1000)])

fake_df = pd.concat([data3, fake_rows], ignore_index = True)

fake_df = fake_df.drop(columns = ['Row ID', 'Product Name', 'Country', 'Postal Code', 'Order ID', 'Order Date'])

fake_df_encoded = pd.get_dummies(fake_df, columns = ['Ship Mode', 'Segment', 'State', 'Region', 'Category', 'Sub-Category'], dtype = int)

X = fake_df_encoded.drop(columns = ['Customer ID', 'Product ID', 'City', 'Purchased'])
y = fake_df_encoded['Purchased']

X.head()

,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,...,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,48.896,4,0.2,8.5568,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
1,474.430,11,0.0,199.2606,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
2,3.600,2,0.0,1.7280,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,454.560,5,0.2,-107.9580,0,0,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0
4,141.420,5,0.6,-187.3815,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
